# Experiment 5— AutoGluon on the preprocessed feature set

This experiment uses `exploration/metul/preprocessing.py` for all feature engineering. Categorical columns remain native categorical values; there is no manual label encoding. The validation split is chronological so later rents are not used to train or tune the validation model.

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'), Path('/content/drive/MyDrive/KDDM-project')]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))
from preprocessing import RentalPreprocessor, load_datasets

try:
    from autogluon.tabular import TabularPredictor
except ImportError as exc:
    raise ImportError('AutoGluon is required. Install it in the active environment with: pip install autogluon.tabular') from exc

RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'
TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_TIME_LIMIT', '600'))
data_dir, datasets = load_datasets(project_root / 'data')
raw_train_df = datasets['train'].copy()
raw_test_df = datasets['test'].copy()
raw_periods = pd.to_datetime(raw_train_df['RENT_APPROVAL_DATE'] + '-01').sort_values().unique()
cutoff = raw_periods[int(len(raw_periods) * 0.80)]
is_train_raw = pd.to_datetime(raw_train_df['RENT_APPROVAL_DATE'] + '-01') < cutoff

# Validation preprocessing is fitted only on earlier rows.
validation_preprocessor = RentalPreprocessor(datasets)
train_df = validation_preprocessor.fit_transform(raw_train_df.loc[is_train_raw])
valid_df = validation_preprocessor.transform(raw_train_df.loc[~is_train_raw])

# The final preprocessor is separately fitted on every labeled row.
final_preprocessor = RentalPreprocessor(datasets)
full_train_df = final_preprocessor.fit_transform(raw_train_df)
test_df = final_preprocessor.transform(raw_test_df)
print(f'Chronological cutoff: {pd.Timestamp(cutoff).date()}')
print(f'Validation train: {train_df.shape}; validation: {valid_df.shape}')
print(f'Final train: {full_train_df.shape}; test: {test_df.shape}')

Chronological cutoff: 2024-05-01
Validation train: (117988, 34); validation: (32012, 34)
Final train: (150000, 34); test: (50000, 33)


## Prepare native feature types

All columns produced by the preprocessing module are retained except the target. The approval-month string is converted to a datetime because AutoGluon can derive calendar features from it. Other text columns are pandas categoricals, allowing AutoGluon to handle them natively.

In [2]:
def prepare_features(frame):
    result = frame.drop(columns=[TARGET], errors='ignore').copy()
    if 'RENT_APPROVAL_DATE' in result:
        result['RENT_APPROVAL_DATE'] = pd.to_datetime(result['RENT_APPROVAL_DATE'] + '-01')
    return result

def align_native_categories(fit_features, transform_features):
    fit_result = fit_features.copy()
    transform_result = transform_features.copy()
    for column in fit_result.select_dtypes(include=['object', 'string', 'category']).columns:
        categories = fit_result[column].fillna('Unknown').astype(str).unique()
        fit_result[column] = pd.Categorical(
            fit_result[column].fillna('Unknown').astype(str), categories=categories
        )
        transform_result[column] = pd.Categorical(
            transform_result[column].fillna('Unknown').astype(str), categories=categories
        )
    return fit_result, transform_result

# Validation categories are learned from the earlier training partition only.
X_train, X_valid = align_native_categories(
    prepare_features(train_df), prepare_features(valid_df)
)
y_train = train_df[TARGET].astype(float)
y_valid = valid_df[TARGET].astype(float)

# Final-model categories are learned from all labeled rows, never the test set.
X_full, X_test = align_native_categories(
    prepare_features(full_train_df), prepare_features(test_df)
)
y_full = full_train_df[TARGET].astype(float)
print(f'Model rows: {len(X_train):,}; validation rows: {len(X_valid):,}')
print(f'Categorical columns: {X_train.select_dtypes(include=["category"]).columns.tolist()}')

Model rows: 117,988; validation rows: 32,012
Categorical columns: ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FURNISHED', 'REGION']


## Validation fit with explicit HPO and evaluation

`tuning_data` is explicitly supplied as the later-period holdout. AutoGluon performs explicit HPO for LightGBM and CatBoost, then ensembles the tuned candidates. The 50 HPO trials are split as 25 trials per model family.

In [3]:
from autogluon.common import space

HPO_TRIALS_PER_MODEL = 25
HPO_HYPERPARAMETERS = {
    'GBM': {
        'num_leaves': space.Int(16, 128),
        'learning_rate': space.Real(0.01, 0.20, log=True),
        'min_data_in_leaf': space.Int(10, 200),
        'feature_fraction': space.Real(0.6, 1.0),
        'bagging_fraction': space.Real(0.6, 1.0),
    },
    'CAT': {
        'depth': space.Int(4, 10),
        'learning_rate': space.Real(0.01, 0.20, log=True),
        'l2_leaf_reg': space.Real(1.0, 20.0, log=True),
    },
}
HPO_KWARGS = {
    'num_trials': HPO_TRIALS_PER_MODEL,
    'scheduler': 'local',
    'searcher': 'random',
}

validation_predictor = TabularPredictor(
    label=TARGET,
    problem_type='regression',
    eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_model_validation_hpo_1_6_2'),
)
validation_train = X_train.copy()
validation_train[TARGET] = y_train.to_numpy()
validation_data = X_valid.copy()
validation_data[TARGET] = y_valid.to_numpy()
validation_predictor.fit(
    train_data=validation_train,
    tuning_data=validation_data,
    hyperparameters=HPO_HYPERPARAMETERS,
    hyperparameter_tune_kwargs=HPO_KWARGS,
    presets='medium_quality',
    time_limit=TIME_LIMIT_SECONDS,
    verbosity=2,
)
valid_predictions = validation_predictor.predict(X_valid)
validation_metrics = pd.DataFrame([{
    'RMSE': mean_squared_error(y_valid, valid_predictions) ** 0.5,
    'MAE': mean_absolute_error(y_valid, valid_predictions),
    'R2': r2_score(y_valid, valid_predictions),
}])
display(validation_metrics)
display(validation_predictor.leaderboard(validation_data, silent=True))
print(f'HPO trials requested: {HPO_TRIALS_PER_MODEL * len(HPO_HYPERPARAMETERS)} total ({HPO_TRIALS_PER_MODEL} per model family)')

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.66 GB / 16.00 GB (35.3%)
Disk Space Avail:   201.02 GB / 460.38 GB (43.7%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_model_validation_hpo_1_6_2"
Train Data Rows:    117988
Train Data Columns: 33
Tuning Data Rows:    32012
Tuning Data Columns: 33
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Featur

[1000]	valid_set's rmse: 478.083
[2000]	valid_set's rmse: 476.81
[3000]	valid_set's rmse: 476.616


 16%|█▌        | 4/25 [00:26<01:55,  5.48s/it]

[1000]	valid_set's rmse: 479.909


100%|██████████| 25/25 [02:42<00:00,  6.50s/it]
Fitted model: LightGBM/T1 ...
	-476.5919	 = Validation score   (-root_mean_squared_error)
	14.58s	 = Training   runtime
	0.65s	 = Validation runtime
Fitted model: LightGBM/T2 ...
	-477.8005	 = Validation score   (-root_mean_squared_error)
	1.66s	 = Training   runtime
	0.06s	 = Validation runtime
Fitted model: LightGBM/T3 ...
	-477.3649	 = Validation score   (-root_mean_squared_error)
	4.13s	 = Training   runtime
	0.18s	 = Validation runtime
Fitted model: LightGBM/T4 ...
	-482.0812	 = Validation score   (-root_mean_squared_error)
	4.6s	 = Training   runtime
	0.14s	 = Validation runtime
Fitted model: LightGBM/T5 ...
	-479.7154	 = Validation score   (-root_mean_squared_error)
	19.38s	 = Training   runtime
	0.79s	 = Validation runtime
Fitted model: LightGBM/T6 ...
	-483.7993	 = Validation score   (-root_mean_squared_error)
	1.62s	 = Training   runtime
	0.03s	 = Validation runtime
Fitted model: LightGBM/T7 ...
	-480.9176	 = Validation score   

,RMSE,MAE,R2
0,469.23363,348.51042,0.489897


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-469.233630,-469.233630,root_mean_squared_error,0.246963,0.211563,147.295113,0.001671,0.000348,0.019087,2,True,27
1,CatBoost/T1,-472.321653,-472.321653,root_mean_squared_error,0.104558,0.107039,143.921030,0.104558,0.107039,143.921030,1,True,26
2,LightGBM/T1,-476.591897,-476.591897,root_mean_squared_error,1.038261,0.652341,14.584989,1.038261,0.652341,14.584989,1,True,1
3,LightGBM/T14,-476.955994,-476.955994,root_mean_squared_error,0.143067,0.101242,2.529106,0.143067,0.101242,2.529106,1,True,14
4,LightGBM/T3,-477.364897,-477.364897,root_mean_squared_error,0.276244,0.183306,4.128654,0.276244,0.183306,4.128654,1,True,3
5,LightGBM/T2,-477.800508,-477.800508,root_mean_squared_error,0.091751,0.057250,1.663407,0.091751,0.057250,1.663407,1,True,2
6,LightGBM/T15,-479.324570,-479.324570,root_mean_squared_error,0.576964,0.416369,9.825302,0.576964,0.416369,9.825302,1,True,15
7,LightGBM/T21,-479.449580,-479.449580,root_mean_squared_error,0.140734,0.104176,3.354996,0.140734,0.104176,3.354996,1,True,21
8,LightGBM/T23,-479.639977,-479.639977,root_mean_squared_error,0.023667,0.017062,0.683659,0.023667,0.017062,0.683659,1,True,23
9,LightGBM/T25,-479.693882,-479.693882,root_mean_squared_error,0.306285,0.234936,6.220183,0.306285,0.234936,6.220183,1,True,25


HPO trials requested: 50 total (25 per model family)


## Final fit and test predictions

After validation is recorded, a separate AutoGluon predictor is fit on every labeled training row. The test set is never used during fitting; it is used only for prediction.

In [4]:
full_training_data = X_full.copy()
full_training_data[TARGET] = y_full.to_numpy()
final_predictor = TabularPredictor(
    label=TARGET,
    problem_type='regression',
    eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_model_final_hpo_1_6_2'),
)
final_predictor.fit(
    train_data=full_training_data,
    hyperparameters=HPO_HYPERPARAMETERS,
    hyperparameter_tune_kwargs=HPO_KWARGS,
    presets='medium_quality',
    time_limit=TIME_LIMIT_SECONDS,
    verbosity=2,
)
test_predictions = final_predictor.predict(X_test).astype(float).clip(lower=0)
submission = test_df.copy()
submission['PREDICTED_MONTHLY_RENT'] = test_predictions.to_numpy()
output_path = experiment_dir / 'exp_5_autogluon_predictions.csv'
submission.to_csv(output_path, index=False)
print(f'Saved {len(submission):,} predictions to {output_path}')
display(submission[['RENT_APPROVAL_DATE', 'TOWN', 'FLAT_TYPE', 'PREDICTED_MONTHLY_RENT']].head())

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.74 GB / 16.00 GB (35.9%)
Disk Space Avail:   200.93 GB / 460.38 GB (43.6%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_model_final_hpo_1_6_2"
Train Data Rows:    150000
Train Data Columns: 33
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPi

[1000]	valid_set's rmse: 476.455


100%|██████████| 25/25 [02:24<00:00,  5.78s/it]
Fitted model: LightGBM/T1 ...
	-476.1982	 = Validation score   (-root_mean_squared_error)
	9.34s	 = Training   runtime
	0.09s	 = Validation runtime
Fitted model: LightGBM/T2 ...
	-477.7567	 = Validation score   (-root_mean_squared_error)
	2.03s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T3 ...
	-477.7041	 = Validation score   (-root_mean_squared_error)
	4.1s	 = Training   runtime
	0.04s	 = Validation runtime
Fitted model: LightGBM/T4 ...
	-482.6618	 = Validation score   (-root_mean_squared_error)
	4.49s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: LightGBM/T5 ...
	-480.1311	 = Validation score   (-root_mean_squared_error)
	12.52s	 = Training   runtime
	0.08s	 = Validation runtime
Fitted model: LightGBM/T6 ...
	-482.9529	 = Validation score   (-root_mean_squared_error)
	1.79s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBM/T7 ...
	-480.5175	 = Validation score   (

Saved 50,000 predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_5_autogluon_predictions.csv


,RENT_APPROVAL_DATE,TOWN,FLAT_TYPE,PREDICTED_MONTHLY_RENT
0,2025-03,kallang/whampoa,3-room,2744.400635
1,2025-03,bukit timah,4-room,3778.836670
2,2025-03,pasir ris,4-room,3109.156738
3,2025-03,punggol,4-room,3144.895752
4,2025-03,tampines,4-room,3267.840332


In [5]:
from autogluon.tabular import TabularPredictor
import pandas as pd

predictor = TabularPredictor.load(
    str(experiment_dir / 'ag_model_final_hpo_1_6_2')
)

predictions = predictor.predict(X_test).astype(float)

submission = pd.DataFrame({
    "Id": test_df.index,
    "Predicted": predictions.to_numpy(),
})

output_path = experiment_dir / "exp_5_autogluon_predictions.csv"
submission.to_csv(output_path, index=False)

print(f"Saved predictions to: {output_path}")
display(submission.head())

Saved predictions to: /Users/metul/Desktop/KDDM-project/exploration/metul/exp_5_autogluon_predictions.csv


,Id,Predicted
0,0,2744.400635
1,1,3778.836670
2,2,3109.156738
3,3,3144.895752
4,4,3267.840332
